# RL Project: Atari Tennis Tournament (NumPy Agents)

This notebook implements four Reinforcement Learning algorithms **without PyTorch** to play Atari Tennis (`ALE/Tennis-v5` via Gymnasium):

1. **SARSA** — Semi-gradient SARSA with linear approximation (inspired by Lab 7, on-policy update from Lab 5B)
2. **Q-Learning** — Off-policy linear approximation (inspired by Lab 5B)
3. **DQN** — Deep Q-Network with pure numpy MLP, experience replay and target network (inspired by Lab 6A + classic DQN)
4. **Monte Carlo** — First-visit MC control with linear approximation (inspired by Lab 4)

Each agent is **pre-trained independently** against the built-in Atari AI opponent, then evaluated in a comparative tournament.

In [17]:
import pickle
from pathlib import Path

import ale_py  # noqa: F401 — registers ALE environments
import gymnasium as gym
from gymnasium.wrappers import FrameStackObservation, ResizeObservation
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns


In [18]:
CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


# Utility Functions

## Observation Normalization

The Tennis environment produces image observations of shape `(4, 84, 84)` after preprocessing (grayscale + resize + frame stack).
We normalize them into 1D `float64` vectors divided by 255, as in Lab 7 (continuous feature normalization).

## ε-greedy Policy

Follows the pattern from Lab 5B (`epsilon_greedy`) and Lab 7 (`epsilon_greedy_action`):
- With probability ε: random action (exploration)
- With probability 1−ε: action maximizing $\hat{q}(s, a)$ with uniform tie-breaking (`np.flatnonzero`)

In [19]:
def normalize_obs(observation: np.ndarray) -> np.ndarray:
    """Flatten and normalize an observation to a 1D float64 vector.

    Replicates the /255.0 normalization used in all agents from the original project.
    For image observations of shape (4, 84, 84), this produces a vector of length 28_224.

    Args:
        observation: Raw observation array from the environment.

    Returns:
        1D numpy array of dtype float64, values in [0, 1].

    """
    return observation.flatten().astype(np.float64) / 255.0


def epsilon_greedy(
    q_values: np.ndarray,
    epsilon: float,
    rng: np.random.Generator,
) -> int:
    """Select an action using an ε-greedy policy with fair tie-breaking.

    Follows the same logic as Lab 5B epsilon_greedy and Lab 7 epsilon_greedy_action:
    - With probability epsilon: choose a random action (exploration).
    - With probability 1-epsilon: choose the action with highest Q-value (exploitation).
    - If multiple actions share the maximum Q-value, break ties uniformly at random.

    Handles edge cases: empty q_values, NaN/Inf values.

    Args:
        q_values: Array of Q-values for each action, shape (n_actions,).
        epsilon: Exploration probability in [0, 1].
        rng: NumPy random number generator.

    Returns:
        Selected action index.

    """
    q_values = np.asarray(q_values, dtype=np.float64).reshape(-1)

    if q_values.size == 0:
        msg = "q_values is empty."
        raise ValueError(msg)

    if rng.random() < epsilon:
        return int(rng.integers(0, q_values.size))

    # Handle NaN/Inf values safely
    finite_mask = np.isfinite(q_values)
    if not np.any(finite_mask):
        return int(rng.integers(0, q_values.size))

    safe_q = q_values.copy()
    safe_q[~finite_mask] = -np.inf
    max_val = np.max(safe_q)
    best = np.flatnonzero(safe_q == max_val)

    if best.size == 0:
        return int(rng.integers(0, q_values.size))

    return int(rng.choice(best))


# Agent Definitions

## Base Class `Agent`

Common interface for all agents, same signatures: `get_action`, `update`, `save`, `load`.
Serialization uses `pickle` (compatible with numpy arrays).

In [20]:
class Agent:
    """Base class for reinforcement learning agents.

    All agents share this interface so they are compatible with the tournament system.
    """

    def __init__(self, seed: int, action_space: int) -> None:
        """Initialize the agent with its action space and a reproducible RNG."""
        self.action_space = action_space
        self.rng = np.random.default_rng(seed=seed)

    def get_action(self, observation: np.ndarray, epsilon: float = 0.0) -> int:
        """Select an action from the current observation."""
        raise NotImplementedError

    def update(
        self,
        state: np.ndarray,
        action: int,
        reward: float,
        next_state: np.ndarray,
        done: bool,
        next_action: int | None = None,
    ) -> None:
        """Update agent parameters from one transition."""

    def save(self, filename: str) -> None:
        """Save the agent state to disk using pickle."""
        with Path(filename).open("wb") as f:
            pickle.dump(self.__dict__, f)

    def load(self, filename: str) -> None:
        """Load the agent state from disk."""
        with Path(filename).open("rb") as f:
            self.__dict__.update(pickle.load(f))  # noqa: S301


## Random Agent (baseline)

Serves as a reference to evaluate the performance of learning agents.

In [21]:
class RandomAgent(Agent):
    """A simple agent that selects actions uniformly at random (baseline)."""

    def get_action(self, observation: np.ndarray, epsilon: float = 0.0) -> int:
        """Select a random action, ignoring the observation and epsilon."""
        _ = observation, epsilon
        return int(self.rng.integers(0, self.action_space))


## SARSA Agent — Linear Approximation (Semi-gradient)

This agent combines:
- **Linear approximation** from Lab 7 (`SarsaAgent`): $\hat{q}(s, a; \mathbf{W}) = \mathbf{W}_a^\top \phi(s)$
- **On-policy SARSA update** from Lab 5B (`train_sarsa`): $\delta = r + \gamma \hat{q}(s', a') - \hat{q}(s, a)$

The semi-gradient update rule is:
$$W_a \leftarrow W_a + \alpha \cdot \delta \cdot \phi(s)$$

where $\phi(s)$ is the normalized observation vector (analogous to tile coding features in Lab 7, but in dense form).

In [22]:
class SarsaAgent(Agent):
    """Semi-gradient SARSA agent with linear function approximation.

    Inspired by:
    - Lab 7 SarsaAgent: linear q(s,a) = W_a . phi(s), semi-gradient update
    - Lab 5B train_sarsa: on-policy TD target using Q(s', a')

    The weight matrix W has shape (n_actions, n_features).
    For a given state s, q(s, a) = W[a] @ phi(s) is the dot product
    of the action's weight row with the normalized observation.
    """

    def __init__(
        self,
        n_features: int,
        n_actions: int,
        alpha: float = 0.001,
        gamma: float = 0.99,
        seed: int = 42,
    ) -> None:
        """Initialize SARSA agent with linear weights.

        Args:
            n_features: Dimension of the feature vector phi(s).
            n_actions: Number of discrete actions.
            alpha: Learning rate (kept small for high-dim features).
            gamma: Discount factor.
            seed: RNG seed for reproducibility.

        """
        super().__init__(seed, n_actions)
        self.n_features = n_features
        self.alpha = alpha
        self.gamma = gamma
        # Weight matrix: one row per action, analogous to Lab 7's self.w
        # but organized as (n_actions, n_features) for dense features.
        self.W = np.zeros((n_actions, n_features), dtype=np.float64)

    def _q_values(self, phi: np.ndarray) -> np.ndarray:
        """Compute Q-values for all actions given feature vector phi(s).

        Equivalent to Lab 7's self.q(s, a) = self.w[idx].sum()
        but using dense linear approximation: q(s, a) = W[a] @ phi.

        Args:
            phi: Normalized feature vector, shape (n_features,).

        Returns:
            Array of Q-values, shape (n_actions,).

        """
        return self.W @ phi  # shape (n_actions,)

    def get_action(self, observation: np.ndarray, epsilon: float = 0.0) -> int:
        """Select action using ε-greedy policy over linear Q-values.

        Same pattern as Lab 7 SarsaAgent.eps_greedy:
        compute q-values for all actions, then apply epsilon_greedy.
        """
        phi = normalize_obs(observation)
        q_vals = self._q_values(phi)
        return epsilon_greedy(q_vals, epsilon, self.rng)

    def update(
        self,
        state: np.ndarray,
        action: int,
        reward: float,
        next_state: np.ndarray,
        done: bool,
        next_action: int | None = None,
    ) -> None:
        """Perform one semi-gradient SARSA update.

        Follows the SARSA update from Lab 5B train_sarsa:
            td_target = r + gamma * Q(s', a') * (0 if done else 1)
            Q(s, a) += alpha * (td_target - Q(s, a))

        In continuous form with linear approximation (Lab 7 SarsaAgent.update):
            delta = target - q(s, a)
            W[a] += alpha * delta * phi(s)

        Args:
            state: Current observation.
            action: Action taken.
            reward: Reward received.
            next_state: Next observation.
            done: Whether the episode ended.
            next_action: Action chosen in next state (required for SARSA).

        """
        phi = np.nan_to_num(normalize_obs(state), nan=0.0, posinf=0.0, neginf=0.0)
        q_sa = float(self.W[action] @ phi)  # current estimate q(s, a)
        if not np.isfinite(q_sa):
            q_sa = 0.0

        if done:
            # Terminal: no future value (Lab 5B: gamma * Q[s2, a2] * 0)
            target = reward
        else:
            # On-policy: use q(s', a') where a' is the actual next action
            # This is the key SARSA property (Lab 5B)
            phi_next = np.nan_to_num(normalize_obs(next_state), nan=0.0, posinf=0.0, neginf=0.0)
            if next_action is None:
                next_action = 0  # fallback, should not happen in practice
            q_sp_ap = float(self.W[next_action] @ phi_next)
            if not np.isfinite(q_sp_ap):
                q_sp_ap = 0.0
            target = float(reward) + self.gamma * q_sp_ap

        # Semi-gradient update: W[a] += alpha * delta * phi(s)
        # Analogous to Lab 7: self.w[idx] += self.alpha * delta
        if not np.isfinite(target):
            return

        delta = float(target - q_sa)
        if not np.isfinite(delta):
            return

        td_step = float(np.clip(delta, -1_000.0, 1_000.0))
        self.W[action] += self.alpha * td_step * phi
        self.W[action] = np.nan_to_num(self.W[action], nan=0.0, posinf=1e6, neginf=-1e6)


## Q-Learning Agent — Linear Approximation (Off-policy)

Same architecture as SARSA but with the **off-policy update** from Lab 5B (`train_q_learning`):

$$\delta = r + \gamma \max_{a'} \hat{q}(s', a') - \hat{q}(s, a)$$

The key difference from SARSA: we use $\max_{a'} Q(s', a')$ instead of $Q(s', a')$ where $a'$ is the action actually chosen. This allows learning the optimal policy independently of the exploration policy.

In [23]:
class QLearningAgent(Agent):
    """Q-Learning agent with linear function approximation (off-policy).

    Inspired by:
    - Lab 5B train_q_learning: off-policy TD target using max_a' Q(s', a')
    - Lab 7 SarsaAgent: linear approximation q(s,a) = W[a] @ phi(s)

    The only difference from SarsaAgent is the TD target:
    SARSA uses Q(s', a') (on-policy), Q-Learning uses max_a' Q(s', a') (off-policy).
    """

    def __init__(
        self,
        n_features: int,
        n_actions: int,
        alpha: float = 0.001,
        gamma: float = 0.99,
        seed: int = 42,
    ) -> None:
        """Initialize Q-Learning agent with linear weights.

        Args:
            n_features: Dimension of the feature vector phi(s).
            n_actions: Number of discrete actions.
            alpha: Learning rate.
            gamma: Discount factor.
            seed: RNG seed.

        """
        super().__init__(seed, n_actions)
        self.n_features = n_features
        self.alpha = alpha
        self.gamma = gamma
        self.W = np.zeros((n_actions, n_features), dtype=np.float64)

    def _q_values(self, phi: np.ndarray) -> np.ndarray:
        """Compute Q-values for all actions: q(s, a) = W[a] @ phi for each a."""
        return self.W @ phi

    def get_action(self, observation: np.ndarray, epsilon: float = 0.0) -> int:
        """Select action using ε-greedy policy over linear Q-values."""
        phi = normalize_obs(observation)
        q_vals = self._q_values(phi)
        return epsilon_greedy(q_vals, epsilon, self.rng)

    def update(
        self,
        state: np.ndarray,
        action: int,
        reward: float,
        next_state: np.ndarray,
        done: bool,
        next_action: int | None = None,
    ) -> None:
        """Perform one Q-learning update.

        Follows Lab 5B train_q_learning:
            td_target = r + gamma * max(Q[s2]) * (0 if terminated else 1)
            Q[s, a] += alpha * (td_target - Q[s, a])

        In continuous form with linear approximation:
            delta = target - q(s, a)
            W[a] += alpha * delta * phi(s)
        """
        _ = next_action  # Q-learning is off-policy: next_action is not used
        phi = np.nan_to_num(normalize_obs(state), nan=0.0, posinf=0.0, neginf=0.0)
        q_sa = float(self.W[action] @ phi)
        if not np.isfinite(q_sa):
            q_sa = 0.0

        if done:
            # Terminal state: no future value
            # Lab 5B: gamma * np.max(Q[s2]) * (0 if terminated else 1)
            target = reward
        else:
            # Off-policy: use max over all actions in next state
            # This is the key Q-learning property (Lab 5B)
            phi_next = np.nan_to_num(normalize_obs(next_state), nan=0.0, posinf=0.0, neginf=0.0)
            q_next_all = self._q_values(phi_next)  # q(s', a') for all a'
            q_next_max = float(np.max(q_next_all))
            if not np.isfinite(q_next_max):
                q_next_max = 0.0
            target = float(reward) + self.gamma * q_next_max

        if not np.isfinite(target):
            return

        delta = float(target - q_sa)
        if not np.isfinite(delta):
            return

        td_step = float(np.clip(delta, -1_000.0, 1_000.0))
        self.W[action] += self.alpha * td_step * phi
        self.W[action] = np.nan_to_num(self.W[action], nan=0.0, posinf=1e6, neginf=-1e6)


## DQN Agent — NumPy MLP with Experience Replay and Target Network

This agent implements the Deep Q-Network (DQN) **entirely in numpy**, without PyTorch.

**Network architecture** (identical to the original DQNAgent structure):
$$\text{Input}(n\_features) \to \text{Linear}(128) \to \text{ReLU} \to \text{Linear}(128) \to \text{ReLU} \to \text{Linear}(n\_actions)$$

**Key techniques** (inspired by Lab 6A Dyna-Q + classic DQN):
- **Experience Replay**: like the `self.model` dict in Dyna-Q (Lab 6A) that stores past transitions, we store transitions in a circular buffer and sample minibatches for updates.
- **Target Network**: periodically synchronized copy of the network, stabilizes learning.
- **Manual backpropagation**: MSE loss gradient backpropagated layer by layer.

In [24]:
def _relu(x: np.ndarray) -> np.ndarray:
    """ReLU activation: max(0, x)."""
    return np.maximum(0, x)


def _relu_grad(x: np.ndarray) -> np.ndarray:
    """Compute the ReLU derivative: return 1 where x > 0, else 0."""
    return (x > 0).astype(np.float64)


def _init_layer(
    fan_in: int, fan_out: int, rng: np.random.Generator,
) -> tuple[np.ndarray, np.ndarray]:
    """Initialize a linear layer with He initialization.

    He init: W ~ N(0, sqrt(2/fan_in)) — standard for ReLU networks.

    Args:
        fan_in: Number of input features.
        fan_out: Number of output features.
        rng: Random number generator.

    Returns:
        Weight matrix W of shape (fan_in, fan_out) and bias vector b of shape (fan_out,).

    """
    scale = np.sqrt(2.0 / fan_in)
    W = rng.normal(0, scale, size=(fan_in, fan_out)).astype(np.float64)
    b = np.zeros(fan_out, dtype=np.float64)
    return W, b


class DQNAgent(Agent):
    """Deep Q-Network agent with a 2-hidden-layer MLP implemented in pure numpy.

    Architecture: Input -> Linear(128) -> ReLU -> Linear(128) -> ReLU -> Linear(n_actions)
    Matches the original PyTorch DQNAgent structure.

    Features:
    - Experience replay buffer (like Dyna-Q model in Lab 6A: store past transitions, sample for updates)
    - Target network (periodically synchronized copy for stable targets)
    - Manual forward pass and backpropagation through the MLP
    """

    def __init__(
        self,
        n_features: int,
        n_actions: int,
        lr: float = 0.0001,
        gamma: float = 0.99,
        buffer_size: int = 10000,
        batch_size: int = 64,
        target_update_freq: int = 1000,
        seed: int = 42,
    ) -> None:
        """Initialize DQN agent.

        Args:
            n_features: Input feature dimension.
            n_actions: Number of discrete actions.
            lr: Learning rate for gradient descent.
            gamma: Discount factor.
            buffer_size: Maximum size of the replay buffer.
            batch_size: Minibatch size for updates.
            target_update_freq: Steps between target network syncs.
            seed: RNG seed.

        """
        super().__init__(seed, n_actions)
        self.n_features = n_features
        self.lr = lr
        self.gamma = gamma
        self.buffer_size = buffer_size
        self.batch_size = batch_size
        self.target_update_freq = target_update_freq
        self.update_step = 0

        # Initialize network parameters: 3 layers
        # Layer 1: n_features -> 128
        # Layer 2: 128 -> 128
        # Layer 3: 128 -> n_actions
        self.params = self._init_params(self.rng)

        # Target network: deep copy of params (like Dyna-Q's model copy concept)
        self.target_params = {k: v.copy() for k, v in self.params.items()}

        # Experience replay buffer: list of (s, a, r, s', done) tuples
        # Analogous to Dyna-Q's self.model dict + self.observed_sa in Lab 6A,
        # but storing raw transitions for off-policy sampling
        self.replay_buffer: list[tuple[np.ndarray, int, float, np.ndarray, bool]] = []

    def _init_params(self, rng: np.random.Generator) -> dict[str, np.ndarray]:
        """Initialize all MLP parameters with He initialization."""
        W1, b1 = _init_layer(self.n_features, 128, rng)
        W2, b2 = _init_layer(128, 128, rng)
        W3, b3 = _init_layer(128, self.action_space, rng)
        return {"W1": W1, "b1": b1, "W2": W2, "b2": b2, "W3": W3, "b3": b3}

    def _forward(
        self, x: np.ndarray, params: dict[str, np.ndarray],
    ) -> tuple[np.ndarray, dict[str, np.ndarray]]:
        """Forward pass through the MLP. Returns output and cached activations for backprop.

        Architecture: x -> Linear -> ReLU -> Linear -> ReLU -> Linear -> output

        Args:
            x: Input array of shape (batch, n_features) or (n_features,).
            params: Dictionary of network parameters.

        Returns:
            output: Q-values of shape (batch, n_actions) or (n_actions,).
            cache: Dictionary of intermediate values for backpropagation.

        """
        # Layer 1
        z1 = x @ params["W1"] + params["b1"]
        h1 = _relu(z1)
        # Layer 2
        z2 = h1 @ params["W2"] + params["b2"]
        h2 = _relu(z2)
        # Layer 3 (output, no activation)
        out = h2 @ params["W3"] + params["b3"]
        cache = {"x": x, "z1": z1, "h1": h1, "z2": z2, "h2": h2}
        return out, cache

    def _backward(
        self,
        d_out: np.ndarray,
        cache: dict[str, np.ndarray],
        params: dict[str, np.ndarray],
    ) -> dict[str, np.ndarray]:
        """Backward pass: compute gradients of loss w.r.t. all parameters.

        Args:
            d_out: Gradient of loss w.r.t. output, shape (batch, n_actions).
            cache: Cached activations from forward pass.
            params: Current network parameters.

        Returns:
            Dictionary of gradients for each parameter.

        """
        batch = d_out.shape[0] if d_out.ndim > 1 else 1
        if d_out.ndim == 1:
            d_out = d_out.reshape(1, -1)

        x = cache["x"] if cache["x"].ndim > 1 else cache["x"].reshape(1, -1)
        h1 = cache["h1"] if cache["h1"].ndim > 1 else cache["h1"].reshape(1, -1)
        h2 = cache["h2"] if cache["h2"].ndim > 1 else cache["h2"].reshape(1, -1)
        z1 = cache["z1"] if cache["z1"].ndim > 1 else cache["z1"].reshape(1, -1)
        z2 = cache["z2"] if cache["z2"].ndim > 1 else cache["z2"].reshape(1, -1)

        # Layer 3 gradients
        dW3 = h2.T @ d_out / batch
        db3 = d_out.mean(axis=0)
        dh2 = d_out @ params["W3"].T

        # Layer 2 gradients (through ReLU)
        dz2 = dh2 * _relu_grad(z2)
        dW2 = h1.T @ dz2 / batch
        db2 = dz2.mean(axis=0)
        dh1 = dz2 @ params["W2"].T

        # Layer 1 gradients (through ReLU)
        dz1 = dh1 * _relu_grad(z1)
        dW1 = x.T @ dz1 / batch
        db1 = dz1.mean(axis=0)

        return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2, "W3": dW3, "b3": db3}

    def get_action(self, observation: np.ndarray, epsilon: float = 0.0) -> int:
        """Select action using ε-greedy policy over MLP Q-values."""
        phi = normalize_obs(observation)
        q_vals, _ = self._forward(phi, self.params)
        if q_vals.ndim > 1:
            q_vals = q_vals.squeeze(0)
        return epsilon_greedy(q_vals, epsilon, self.rng)

    def update(
        self,
        state: np.ndarray,
        action: int,
        reward: float,
        next_state: np.ndarray,
        done: bool,
        next_action: int | None = None,
    ) -> None:
        """Store transition and perform a minibatch DQN update.

        Steps:
        1. Add transition to replay buffer (like Dyna-Q model update in Lab 6A)
        2. If buffer has enough samples, sample a minibatch
        3. Compute targets using target network (Q-learning: max_a' Q_target(s', a'))
        4. Backpropagate MSE loss through the MLP
        5. Update weights with gradient descent
        6. Periodically sync target network

        """
        _ = next_action  # DQN is off-policy

        # Store transition in replay buffer
        # Analogous to Lab 6A: self.model[(s,a)] = (r, sp)
        phi_s = normalize_obs(state)
        phi_sp = normalize_obs(next_state)
        self.replay_buffer.append((phi_s, action, reward, phi_sp, done))
        if len(self.replay_buffer) > self.buffer_size:
            self.replay_buffer.pop(0)

        # Don't update until we have enough samples
        if len(self.replay_buffer) < self.batch_size:
            return

        # Sample minibatch from replay buffer
        # Like Dyna-Q planning: sample from observed transitions (Lab 6A)
        idx = self.rng.choice(len(self.replay_buffer), size=self.batch_size, replace=False)
        batch = [self.replay_buffer[i] for i in idx]

        states_b = np.array([t[0] for t in batch])      # (batch, n_features)
        actions_b = np.array([t[1] for t in batch])      # (batch,)
        rewards_b = np.array([t[2] for t in batch])      # (batch,)
        next_states_b = np.array([t[3] for t in batch])  # (batch, n_features)
        dones_b = np.array([t[4] for t in batch], dtype=np.float64)

        # Forward pass on current states
        q_all, cache = self._forward(states_b, self.params)
        # Gather Q-values for the taken actions
        q_curr = q_all[np.arange(self.batch_size), actions_b]  # (batch,)

        # Compute targets using target network (off-policy: max over actions)
        # Follows Lab 5B Q-learning: td_target = r + gamma * max(Q[s2]) * (0 if terminated else 1)
        q_next_all, _ = self._forward(next_states_b, self.target_params)
        q_next_max = np.max(q_next_all, axis=1)  # (batch,)
        targets = rewards_b + (1.0 - dones_b) * self.gamma * q_next_max

        # MSE loss gradient: d_loss/d_q_curr = 2 * (q_curr - targets) / batch
        # We only backprop through the action that was taken
        d_out = np.zeros_like(q_all)
        d_out[np.arange(self.batch_size), actions_b] = 2.0 * (q_curr - targets) / self.batch_size

        # Backward pass
        grads = self._backward(d_out, cache, self.params)

        # Gradient descent update
        for key in self.params:
            self.params[key] -= self.lr * grads[key]

        # Sync target network periodically
        self.update_step += 1
        if self.update_step % self.target_update_freq == 0:
            self.target_params = {k: v.copy() for k, v in self.params.items()}


## Monte Carlo Agent — Linear Approximation (First-visit)

This agent is inspired by Lab 4 (`mc_control_epsilon_soft`):
- Accumulates transitions in an episode buffer `(state, action, reward)`
- At the end of the episode (`done=True`), computes **cumulative returns** by traversing the buffer backward:
  $$G \leftarrow \gamma \cdot G + r$$
- Updates weights with the semi-gradient rule:
  $$W_a \leftarrow W_a + \alpha \cdot (G - \hat{q}(s, a)) \cdot \phi(s)$$

Unlike TD methods (SARSA, Q-Learning), Monte Carlo waits for the complete episode to finish before updating.

In [25]:
class MonteCarloAgent(Agent):
    """Monte Carlo control agent with linear function approximation.

    Inspired by Lab 4 mc_control_epsilon_soft:
    - Accumulates transitions in an episode buffer
    - At episode end (done=True), computes discounted returns backward:
        G = gamma * G + r  (same as Lab 4's reversed loop)
    - Updates weights with semi-gradient: W[a] += alpha * (G - q(s,a)) * phi(s)

    Unlike TD methods (SARSA, Q-Learning), no update occurs until the episode ends.
    """

    def __init__(
        self,
        n_features: int,
        n_actions: int,
        alpha: float = 0.001,
        gamma: float = 0.99,
        seed: int = 42,
    ) -> None:
        """Initialize Monte Carlo agent.

        Args:
            n_features: Dimension of the feature vector phi(s).
            n_actions: Number of discrete actions.
            alpha: Learning rate.
            gamma: Discount factor.
            seed: RNG seed.

        """
        super().__init__(seed, n_actions)
        self.n_features = n_features
        self.alpha = alpha
        self.gamma = gamma
        self.W = np.zeros((n_actions, n_features), dtype=np.float64)
        # Episode buffer: stores (state, action, reward) tuples
        # Analogous to Lab 4's episode list in generate_episode
        self.episode_buffer: list[tuple[np.ndarray, int, float]] = []

    def _q_values(self, phi: np.ndarray) -> np.ndarray:
        """Compute Q-values for all actions: q(s, a) = W[a] @ phi for each a."""
        return self.W @ phi

    def get_action(self, observation: np.ndarray, epsilon: float = 0.0) -> int:
        """Select action using ε-greedy policy over linear Q-values."""
        phi = normalize_obs(observation)
        q_vals = self._q_values(phi)
        return epsilon_greedy(q_vals, epsilon, self.rng)

    def update(
        self,
        state: np.ndarray,
        action: int,
        reward: float,
        next_state: np.ndarray,
        done: bool,
        next_action: int | None = None,
    ) -> None:
        """Accumulate transitions and update at episode end with MC returns.

        Follows Lab 4 mc_control_epsilon_soft / mc_control_exploring_starts:
        1. Append (state, action, reward) to episode buffer
        2. If not done: wait (no update yet)
        3. If done: compute returns backward and update weights

        The backward loop is exactly the Lab 4 pattern:
            G = 0
            for s, a, r in reversed(episode_buffer):
                G = gamma * G + r
                # update Q(s, a) toward G
        """
        _ = next_state, next_action  # Not used in MC

        self.episode_buffer.append((state, action, reward))

        if not done:
            return  # Wait until episode ends

        # Episode finished: compute MC returns and update
        # Backward pass through episode (Lab 4 pattern)
        returns = 0.0
        for s, a, r in reversed(self.episode_buffer):
            returns = self.gamma * returns + r

            phi = np.nan_to_num(normalize_obs(s), nan=0.0, posinf=0.0, neginf=0.0)
            q_sa = float(self.W[a] @ phi)
            if not np.isfinite(q_sa):
                q_sa = 0.0

            # Semi-gradient update toward the MC return G
            # Analogous to Lab 4: Q[(s,a)] += (G - Q[(s,a)]) / N[(s,a)]
            # but with linear approximation and fixed step size
            if not np.isfinite(returns):
                continue

            delta = float(returns - q_sa)
            if not np.isfinite(delta):
                continue

            td_step = float(np.clip(delta, -1_000.0, 1_000.0))
            self.W[a] += self.alpha * td_step * phi
            self.W[a] = np.nan_to_num(self.W[a], nan=0.0, posinf=1e6, neginf=-1e6)

        # Clear episode buffer for next episode
        self.episode_buffer = []


## Tennis Environment

Creation of the Atari Tennis environment via Gymnasium (`ALE/Tennis-v5`) with standard wrappers:
- **Grayscale**: `obs_type="grayscale"` — single-channel observations
- **Resize**: `ResizeObservation(84, 84)` — downscale to 84×84
- **Frame stack**: `FrameStackObservation(4)` — stack 4 consecutive frames

The final observation is an array of shape `(4, 84, 84)`, which flattens to 28,224 features.

The agent plays against the **built-in Atari AI opponent**.

In [26]:
def create_env() -> gym.Env:
    """Create the ALE/Tennis-v5 environment with preprocessing wrappers.

    Applies:
    - obs_type="grayscale": grayscale observation (210, 160)
    - ResizeObservation(84, 84): downscale to 84x84
    - FrameStackObservation(4): stack 4 consecutive frames -> (4, 84, 84)

    Returns:
        Gymnasium environment ready for training.

    """
    env = gym.make("ALE/Tennis-v5", obs_type="grayscale")
    env = ResizeObservation(env, shape=(84, 84))
    return FrameStackObservation(env, stack_size=4)


## Training & Evaluation Infrastructure

Functions for training and evaluating agents in the single-agent Gymnasium environment:

1. **`train_agent`** — Pre-trains an agent against the built-in AI for a given number of episodes with ε-greedy exploration
2. **`evaluate_agent`** — Evaluates a trained agent (no exploration, ε = 0) and returns performance metrics
3. **`plot_training_curves`** — Plots the training reward history (moving average) for all agents
4. **`plot_evaluation_comparison`** — Bar chart comparing final evaluation scores across agents
5. **`evaluate_tournament`** — Evaluates all agents and produces a summary comparison

In [27]:
def train_agent(
    env: gym.Env,
    agent: Agent,
    name: str,
    *,
    episodes: int = 5000,
    epsilon_start: float = 1.0,
    epsilon_end: float = 0.05,
    epsilon_decay: float = 0.999,
    max_steps: int = 5000,
) -> list[float]:
    """Pre-train an agent against the built-in Atari AI opponent.

    Each agent learns independently by playing full episodes. This is the
    self-play pre-training phase: the agent interacts with the environment's
    built-in opponent and updates its parameters after each transition.

    Args:
        env: Gymnasium ALE/Tennis-v5 environment.
        agent: Agent instance to train.
        name: Display name for the progress bar.
        episodes: Number of training episodes.
        epsilon_start: Initial exploration rate.
        epsilon_end: Minimum exploration rate.
        epsilon_decay: Multiplicative decay per episode.
        max_steps: Maximum steps per episode.

    Returns:
        List of total rewards per episode.

    """
    rewards_history: list[float] = []
    epsilon = epsilon_start

    pbar = tqdm(range(episodes), desc=f"Training {name}", leave=True)

    for _ep in pbar:
        obs, _info = env.reset()
        obs = np.asarray(obs)
        total_reward = 0.0

        # Select first action
        action = agent.get_action(obs, epsilon=epsilon)

        for _step in range(max_steps):
            next_obs, reward, terminated, truncated, _info = env.step(action)
            next_obs = np.asarray(next_obs)
            done = terminated or truncated
            reward = float(reward)
            total_reward += reward

            # Select next action (needed for SARSA's on-policy update)
            next_action = agent.get_action(next_obs, epsilon=epsilon) if not done else None

            # Update agent with the transition
            agent.update(
                state=obs,
                action=action,
                reward=reward,
                next_state=next_obs,
                done=done,
                next_action=next_action,
            )

            if done:
                break

            obs = next_obs
            action = next_action

        rewards_history.append(total_reward)
        epsilon = max(epsilon_end, epsilon * epsilon_decay)

        # Update progress bar
        recent_window = 50
        if len(rewards_history) >= recent_window:
            recent_avg = np.mean(rewards_history[-recent_window:])
            pbar.set_postfix(
            avg50=f"{recent_avg:.1f}",
            eps=f"{epsilon:.3f}",
            rew=f"{total_reward:.0f}",
            )

    return rewards_history


def evaluate_agent(
    env: gym.Env,
    agent: Agent,
    name: str,
    *,
    episodes: int = 20,
    max_steps: int = 5000,
) -> dict[str, object]:
    """Evaluate a trained agent with no exploration (ε = 0).

    Args:
        env: Gymnasium ALE/Tennis-v5 environment.
        agent: Trained agent to evaluate.
        name: Display name for the progress bar.
        episodes: Number of evaluation episodes.
        max_steps: Maximum steps per episode.

    Returns:
        Dictionary with rewards list, mean, std, wins, and win rate.

    """
    rewards: list[float] = []
    wins = 0

    for _ep in tqdm(range(episodes), desc=f"Evaluating {name}", leave=False):
        obs, _info = env.reset()
        total_reward = 0.0

        for _step in range(max_steps):
            action = agent.get_action(np.asarray(obs), epsilon=0.0)
            obs, reward, terminated, truncated, _info = env.step(action)
            reward = float(reward)
            total_reward += reward
            if terminated or truncated:
                break

        rewards.append(total_reward)
        if total_reward > 0:
            wins += 1

    return {
        "rewards": rewards,
        "mean_reward": float(np.mean(rewards)),
        "std_reward": float(np.std(rewards)),
        "wins": wins,
        "win_rate": wins / episodes,
    }


def plot_training_curves(
    training_histories: dict[str, list[float]],
    window: int = 100,
) -> None:
    """Plot training reward curves for all agents on a single figure.

    Uses a moving average to smooth the curves.

    Args:
        training_histories: Dict mapping agent names to reward lists.
        window: Moving average window size.

    """
    plt.figure(figsize=(12, 6))

    for name, rewards in training_histories.items():
        if len(rewards) >= window:
            ma = np.convolve(rewards, np.ones(window) / window, mode="valid")
            plt.plot(np.arange(window - 1, len(rewards)), ma, label=name)
        else:
            plt.plot(rewards, label=f"{name} (raw)")

    plt.xlabel("Episodes")
    plt.ylabel(f"Average Reward (Window={window})")
    plt.title("Training Curves (vs built-in AI)")
    plt.legend()
    plt.grid(visible=True)
    plt.tight_layout()
    plt.show()


def plot_evaluation_comparison(results: dict[str, dict[str, object]]) -> None:
    """Bar chart comparing evaluation performance of all agents.

    Args:
        results: Dict mapping agent names to evaluation result dicts.

    """
    names = list(results.keys())
    means = [results[n]["mean_reward"] for n in names]
    stds = [results[n]["std_reward"] for n in names]
    win_rates = [results[n]["win_rate"] for n in names]

    _fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Mean reward bar chart
    colors = sns.color_palette("husl", len(names))
    axes[0].bar(names, means, yerr=stds, capsize=5, color=colors, edgecolor="black")
    axes[0].set_ylabel("Mean Reward")
    axes[0].set_title("Evaluation: Mean Reward per Agent (vs built-in AI)")
    axes[0].axhline(y=0, color="gray", linestyle="--", alpha=0.5)
    axes[0].grid(axis="y", alpha=0.3)

    # Win rate bar chart
    axes[1].bar(names, win_rates, color=colors, edgecolor="black")
    axes[1].set_ylabel("Win Rate")
    axes[1].set_title("Evaluation: Win Rate per Agent (vs built-in AI)")
    axes[1].set_ylim(0, 1)
    axes[1].axhline(y=0.5, color="gray", linestyle="--", alpha=0.5, label="50% baseline")
    axes[1].legend()
    axes[1].grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()


def evaluate_tournament(
    env: gym.Env,
    agents: dict[str, Agent],
    episodes_per_agent: int = 20,
) -> dict[str, dict[str, object]]:
    """Evaluate all agents against the built-in AI and produce a comparison.

    Args:
        env: Gymnasium ALE/Tennis-v5 environment.
        agents: Dictionary mapping agent names to Agent instances.
        episodes_per_agent: Number of evaluation episodes per agent.

    Returns:
        Dict mapping agent names to their evaluation results.

    """
    results: dict[str, dict[str, object]] = {}
    n_agents = len(agents)

    for idx, (name, agent) in enumerate(agents.items(), start=1):
        print(f"[Evaluation {idx}/{n_agents}] {name}")
        results[name] = evaluate_agent(
            env, agent, name, episodes=episodes_per_agent,
        )
        mean_r = results[name]["mean_reward"]
        wr = results[name]["win_rate"]
        print(f"  -> Mean reward: {mean_r:.2f} | Win rate: {wr:.1%}\n")

    return results


## Agent Instantiation & Incremental Training (One Agent at a Time)

**Environment**: `ALE/Tennis-v5` (grayscale, 84×84×4 frames → 28,224 features, 18 actions).

**Agents**:
- **Random** — random baseline (no training needed)
- **SARSA** — linear approximation, semi-gradient TD(0)
- **Q-Learning** — linear approximation, off-policy
- **DQN** — NumPy MLP (2 hidden layers of 128, experience replay, target network)
- **Monte Carlo** — linear approximation, first-visit returns

**Workflow**:
1. Train **one** selected agent (`AGENT_TO_TRAIN`)
2. Save its weights to `checkpoints/<agent>.pkl`
3. Repeat later for another agent without retraining previous ones
4. Load all saved checkpoints before the final evaluation

In [28]:
# Create environment
env = create_env()
obs, _info = env.reset()

n_actions = int(env.action_space.n)
n_features = int(np.prod(obs.shape))

print(f"Observation shape : {obs.shape}")
print(f"Feature vector dim: {n_features}")
print(f"Number of actions : {n_actions}")

# Instantiate agents
agent_random = RandomAgent(seed=42, action_space=int(n_actions))
agent_sarsa = SarsaAgent(n_features=n_features, n_actions=n_actions, alpha=1e-5)
agent_q = QLearningAgent(n_features=n_features, n_actions=n_actions, alpha=1e-5)
agent_dqn = DQNAgent(n_features=n_features, n_actions=n_actions, lr=1e-4)
agent_mc = MonteCarloAgent(n_features=n_features, n_actions=n_actions, alpha=1e-5)

agents = {
    "Random": agent_random,
    "SARSA": agent_sarsa,
    "Q-Learning": agent_q,
    "DQN": agent_dqn,
    "Monte Carlo": agent_mc,
}


Observation shape : (4, 84, 84)
Feature vector dim: 28224
Number of actions : 18


In [ ]:
AGENT_TO_TRAIN = "SARSA"  # change to: "Q-Learning", "DQN", "Monte Carlo", "Random"
TRAINING_EPISODES = 5000
FORCE_RETRAIN = False

if AGENT_TO_TRAIN not in agents:
    msg = f"Unknown agent '{AGENT_TO_TRAIN}'. Available: {list(agents)}"
    raise ValueError(msg)

training_histories: dict[str, list[float]] = {}
agent = agents[AGENT_TO_TRAIN]
ckpt_name = AGENT_TO_TRAIN.lower().replace(" ", "_").replace("-", "_") + ".pkl"
ckpt_path = CHECKPOINT_DIR / ckpt_name

print(f"Selected agent: {AGENT_TO_TRAIN}")
print(f"Checkpoint path: {ckpt_path}")

if AGENT_TO_TRAIN == "Random":
    print("Random is a baseline and is not trained.")
    training_histories[AGENT_TO_TRAIN] = []
elif ckpt_path.exists() and not FORCE_RETRAIN:
    agent.load(str(ckpt_path))
    print("Checkpoint found -> weights loaded, training skipped.")
    training_histories[AGENT_TO_TRAIN] = []
else:
    print(f"\n{'='*60}")
    print(f"Training: {AGENT_TO_TRAIN} ({TRAINING_EPISODES} episodes)")
    print(f"{'='*60}")

    training_histories[AGENT_TO_TRAIN] = train_agent(
        env=env,
        agent=agent,
        name=AGENT_TO_TRAIN,
        episodes=TRAINING_EPISODES,
        epsilon_start=1.0,
        epsilon_end=0.05,
        epsilon_decay=0.999,
    )

    avg_last_100 = np.mean(training_histories[AGENT_TO_TRAIN][-100:])
    print(f"-> {AGENT_TO_TRAIN} avg reward (last 100 eps): {avg_last_100:.2f}")

    agent.save(str(ckpt_path))
    print("Checkpoint saved.")

if training_histories.get(AGENT_TO_TRAIN):
    plot_training_curves(training_histories, window=100)


Selected agent: SARSA
Checkpoint path: checkpoints/sarsa.pkl

Training: SARSA (5000 episodes)


Selected agent: SARSA
Checkpoint path: checkpoints/sarsa.pkl

Training: SARSA (5000 episodes)


Training SARSA:   0%|          | 0/5000 [00:00<?, ?it/s]

In [ ]:
# Load all available checkpoints before final evaluation
CHECKPOINT_DIR = Path("checkpoints")
loaded_agents: list[str] = []
missing_agents: list[str] = []

for name, agent in agents.items():
    if name == "Random":
        continue

    ckpt_name = name.lower().replace(" ", "_").replace("-", "_") + ".pkl"
    ckpt_path = CHECKPOINT_DIR / ckpt_name

    if ckpt_path.exists():
        agent.load(str(ckpt_path))
        loaded_agents.append(name)
    else:
        missing_agents.append(name)

print(f"Loaded checkpoints: {loaded_agents}")
if missing_agents:
    print(f"Missing checkpoints (untrained/not saved yet): {missing_agents}")


## Final Evaluation

Each agent plays 20 episodes against the built-in AI with no exploration (ε = 0).
Performance is compared via mean reward and win rate.

In [ ]:
# Build evaluation set: Random + agents with existing checkpoints
eval_agents: dict[str, Agent] = {"Random": agents["Random"]}
missing_agents: list[str] = []

for name, agent in agents.items():
    if name == "Random":
        continue

    ckpt_name = name.lower().replace(" ", "_").replace("-", "_") + ".pkl"
    ckpt_path = CHECKPOINT_DIR / ckpt_name

    if ckpt_path.exists():
        agent.load(str(ckpt_path))
        eval_agents[name] = agent
    else:
        missing_agents.append(name)

print(f"Agents evaluated: {list(eval_agents.keys())}")
if missing_agents:
    print(f"Skipped (no checkpoint yet): {missing_agents}")

if len(eval_agents) < 2:
    raise RuntimeError("Train at least one non-random agent before final evaluation.")

results = evaluate_tournament(env, eval_agents, episodes_per_agent=20)
plot_evaluation_comparison(results)

# Print summary table
print(f"\n{'Agent':<15} {'Mean Reward':>12} {'Std':>8} {'Win Rate':>10}")
print("-" * 48)
for name, res in results.items():
    print(f"{name:<15} {res['mean_reward']:>12.2f} {res['std_reward']:>8.2f} {res['win_rate']:>9.1%}")
